# sktime-agentic-forecaster — demo

End-to-end walkthrough using the classic airline passengers dataset.
The agent picks the right sktime pipeline from a plain-English description.

**Default LLM:** Gemini `gemini-3.1-flash-lite-preview` via `google-genai` SDK.

> **Step 1:** Run the install cell below. **Step 2:** Paste your Gemini API key. **Step 3:** Run all remaining cells.

In [ ]:
import subprocess, sys, os, importlib

if not os.path.exists("sktime-agentic-forecaster"):
    subprocess.run(
        ["git", "clone", "https://github.com/mohammedfirdouss/sktime-agentic-forecaster.git"],
        check=True
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "-e", "sktime-agentic-forecaster/.[gemini]",
     "matplotlib", "pmdarima"],
    check=True
)

if "sktime-agentic-forecaster" not in sys.path:
    sys.path.insert(0, "sktime-agentic-forecaster")

importlib.invalidate_caches()
print("✓ Installation complete")

In [ ]:
import os
# Get a free key at: https://aistudio.google.com
os.environ["GOOGLE_API_KEY"] = ""  # ← paste your key here

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("sktime-agentic-forecaster"))

import matplotlib.pyplot as plt
from sktime.datasets import load_airline
import pmdarima
from sktime_agent import AgenticForecaster
print("✓ Imports OK")

## 1. Load data

In [ ]:
y = load_airline()   # monthly airline passengers 1949–1960
y_train = y[:-12]    # hold out last 12 months for comparison
y_test  = y[-12:]

print(f"Training: {len(y_train)} obs  |  Test: {len(y_test)} obs")
y_train.plot(title="Airline passengers (training)");

## 2. Run the agentic forecaster

In [ ]:
forecaster = AgenticForecaster(llm_backend="gemini", verbose=True)

result = forecaster.forecast(
    prompt="""
        Forecast the next 12 months of airline passenger numbers.
        The series has a clear upward trend and strong annual seasonality.
        Please use deseasonalisation and a PolynomialDetrender with degree 1 for detrending before fitting.
    """,
    data=y_train,
    horizon=12,
)

## 3. Inspect the result

In [ ]:
print("Selected pipeline:", result.selected_estimators)
print("\nReasoning:")
print(result.explanation)
print("\nPredictions:")
print(result.predictions)

In [ ]:
print(result.pipeline)

## 4. Plot predictions vs actuals

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
y_train.plot(ax=ax, label="Train", color="steelblue")
y_test.plot(ax=ax, label="Actual", color="grey", linestyle="--")
result.predictions.plot(ax=ax, label="Predicted", color="tomato")
ax.set_title(f"Airline passengers — {' → '.join(result.selected_estimators)}")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Re-use the fitted pipeline

The returned `pipeline` is a real sktime estimator — refit it, serialize it, or use it in cross-validation.

In [ ]:
from sktime.forecasting.base import ForecastingHorizon

result.pipeline.fit(y)
extended = result.pipeline.predict(ForecastingHorizon(list(range(1, 7))))
print(extended)

## 6. Swap backends

Change `llm_backend` without touching anything else:

In [ ]:
# Anthropic (Claude)
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# result2 = AgenticForecaster(llm_backend="anthropic").forecast(
#     prompt="...", data=y_train, horizon=12)

# OpenAI
# os.environ["OPENAI_API_KEY"] = "sk-..."
# result3 = AgenticForecaster(llm_backend="openai").forecast(
#     prompt="...", data=y_train, horizon=12)

# LangChain
# from langchain_openai import ChatOpenAI
# result4 = AgenticForecaster(llm_backend=ChatOpenAI(model="gpt-4o")).forecast(
#     prompt="...", data=y_train, horizon=12)